# BIP-39 and BIP-32: Key Derivation by Foot

Companion to [BIP-39 and BIP-32: Understanding Key Derivation](https://weisser-zwerg.dev/posts/bip39-bip32-key-derivation/).

We start with 11-bit word-list indices, construct and validate recovery phrases, and derive the 64-byte BIP-39 seed using Python's standard library.
We then implement BIP-32 child derivation and follow Bitcoin and Ethereum paths, using `coincurve` only for the secp256k1 public-key primitive.
If you prefer the concise library approach, open the [bip_utils notebook](bip39-bip32-key-derivation-bip-utils.ipynb).

In [1]:
%load_ext watermark
%watermark -a 'cs224' -u -d -v -p bip_utils,notebook,nbclassic,watermark,coincurve

Author: cs224

Last updated: 2026-11-26

Python implementation: CPython
Python version       : 3.12.9
IPython version      : 9.19.1

bip_utils: 2.12.3
notebook : 7.6.5
nbclassic: 1.3.3
watermark: 2.6.0
coincurve: 21.0.0



In [2]:
from IPython.display import HTML, display

# Notebook 6 and nbclassic
# https://github.com/jupyter/nbclassic
display(HTML("<style>.container { width:70% !important; }</style>"))

# Notebook 7
display(HTML("<style>:root { --jp-notebook-max-width: 70% !important; }</style>"))

## Public Example Recovery Phrases

We use the public all-zero-entropy examples: `abandon` repeated 11 times followed by `about`, or 23 times followed by `art`.
These predictable phrases are for learning and must never hold funds.

Start with `ENTROPY_BITS = 128` and an empty `BIP39_PASSPHRASE`.
Set `ENTROPY_BITS = 256` and rerun the notebook from the beginning for the 24-word example.

In [3]:
MNEMONIC_12 = "abandon " * 11 + "about"
MNEMONIC_24 = "abandon " * 23 + "art"

ENTROPY_BITS = 128  # Use 256 for the 24-word example.
BIP39_PASSPHRASE = ""

print("12-word public example:", MNEMONIC_12)
print("24-word public example:", MNEMONIC_24)

12-word public example: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about
24-word public example: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon art


## Each Word Represents 11 Bits

[BIP-39](https://github.com/bitcoin/bips/blob/master/bip-0039.mediawiki) uses a defined list of 2,048 words.
Since `2048 = 2 ** 11`, each zero-based index fits into exactly 11 bits.
We read the local `english.txt` copy in its original order, then print a few indices and their binary representations.
The source and license of the list are recorded in the README.

| Index | 11-bit representation | Word |
| --- | --- | --- |
| 0 | `00000000000` | `abandon` |
| 1 | `00000000001` | `ability` |
| 3 | `00000000011` | `about` |
| 102 | `00001100110` | `art` |
| 2047 | `11111111111` | `zoo` |

In [4]:
from pathlib import Path

wordlist = Path("english.txt").read_text(encoding="utf-8").splitlines()
assert len(wordlist) == 2048
assert len(set(wordlist)) == 2048

for index in (0, 1, 3, 102, 2047):
    print(f"{index:4d}  {index:011b}  {wordlist[index]}")

   0  00000000000  abandon
   1  00000000001  ability
   3  00000000011  about
 102  00001100110  art
2047  11111111111  zoo


## Starting Bytes and the Checksum

For our reproducible examples, use all-zero bytes: 16 bytes for 128 bits, or 32 bytes for 256 bits.
Fresh wallet entropy instead needs cryptographically secure random bytes; the commented `secrets.token_bytes()` line shows that operation.
Hashing a predictable input does not make it unpredictable.

Append the first `entropy_bits // 32` bits of SHA-256 to the entropy.
The checksum is at most eight bits long, so we can read it from the first digest byte.
It adds error checking, not fresh randomness.

| Entropy bits | Checksum bits | Total bits | Words |
| --- | --- | --- | --- |
| 128 | 4 | 132 | 12 |
| 160 | 5 | 165 | 15 |
| 192 | 6 | 198 | 18 |
| 224 | 7 | 231 | 21 |
| 256 | 8 | 264 | 24 |

In [5]:
import hashlib
import secrets

entropy = bytes(ENTROPY_BITS // 8)
# For fresh random starting bytes, use:
# entropy = secrets.token_bytes(ENTROPY_BITS // 8)

entropy_bits = "".join(f"{byte:08b}" for byte in entropy)
checksum_length = len(entropy_bits) // 32
checksum_hash = hashlib.sha256(entropy).digest()
checksum_bits = f"{checksum_hash[0]:08b}"[:checksum_length]
bits = entropy_bits + checksum_bits

print("Entropy (hex):", entropy.hex())
print("Entropy bits:", len(entropy_bits))
print("Checksum bits:", checksum_bits)
print("Total bits:", len(bits))

Entropy (hex): 00000000000000000000000000000000
Entropy bits: 128
Checksum bits: 0011
Total bits: 132


## From Bits to Words

Split the combined bits into 11-bit groups, interpret each as an integer, and use that index to select a word.
The `:08b` and `:011b` formats preserve leading zeroes at the byte and word levels.

In [6]:
groups = [bits[offset:offset + 11] for offset in range(0, len(bits), 11)]
indices = [int(group, 2) for group in groups]
mnemonic = " ".join(wordlist[index] for index in indices)

for position, (group, index) in enumerate(zip(groups, indices), start=1):
    print(f"{position:2d}  {group}  {index:4d}  {wordlist[index]}")

print("\nRecovery phrase:", mnemonic)

 1  00000000000     0  abandon
 2  00000000000     0  abandon
 3  00000000000     0  abandon
 4  00000000000     0  abandon
 5  00000000000     0  abandon
 6  00000000000     0  abandon
 7  00000000000     0  abandon
 8  00000000000     0  abandon
 9  00000000000     0  abandon
10  00000000000     0  abandon
11  00000000000     0  abandon
12  00000000011     3  about

Recovery phrase: abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about


### Why the Last Word Is about or art

For 128 zero bits, the last group is seven zero entropy bits plus checksum `0011`: `00000000011`, index 3, or `about`.
For 256 zero bits, it is three zero entropy bits plus checksum `01100110`: `00001100110`, index 102, or `art`.

The earlier groups are all zero and select `abandon`.
The last word contains both entropy and checksum bits; it is not a separate checksum word.
Change `ENTROPY_BITS` above and rerun from the beginning to see both cases.

## Recovering and Validating the Entropy

Reverse the word-list lookup, separate the checksum, and hash the recovered bytes to check it.
For valid phrase lengths, the checksum has `word_count // 3` bits.
The fixed byte length in `to_bytes()` retains leading zero bytes.

An unknown word, invalid count, or checksum mismatch is rejected.
Some changed phrases still have valid checksums; validity alone does not identify the intended wallet.

In [7]:
words = mnemonic.split()
if len(words) not in (12, 15, 18, 21, 24):
    raise ValueError("Invalid BIP-39 word count")

recovered_bits = "".join(f"{wordlist.index(word):011b}" for word in words)
received_checksum_length = len(words) // 3
recovered_entropy_bits = recovered_bits[:-received_checksum_length]
received_checksum = recovered_bits[-received_checksum_length:]

recovered_entropy = int(recovered_entropy_bits, 2).to_bytes(
    len(recovered_entropy_bits) // 8, "big"
)
expected_checksum = f"{hashlib.sha256(recovered_entropy).digest()[0]:08b}"[
    :received_checksum_length
]
if received_checksum != expected_checksum:
    raise ValueError("BIP-39 checksum does not match")

assert recovered_entropy == entropy
print("Recovered entropy:", recovered_entropy.hex())
print("Checksum matches:", received_checksum == expected_checksum)

Recovered entropy: 00000000000000000000000000000000
Checksum matches: True


## From Recovery Phrase to Seed

**PBKDF2 means Password-Based Key Derivation Function 2**, defined in **PKCS #5**.
It is a key derivation function (KDF): the recovery phrase is the starting secret, and the seed is the derived binary key material.
Python implements it in [`hashlib.pbkdf2_hmac(...)`](https://docs.python.org/3.12/library/hashlib.html#hashlib.pbkdf2_hmac).
BIP-39 uses HMAC-SHA512 with 2,048 iterations and a 64-byte output.

The `password` input is the selected recovery phrase, including the final word, with single spaces between the words.
The salt is `mnemonic` followed by `BIP39_PASSPHRASE`; with an empty passphrase it is just `mnemonic`.
Normalize both text inputs using NFKD, then encode them as UTF-8.
The next cell prints their exact byte representations before deriving the seed.

A salt is an additional derivation input; how it is combined with the other inputs depends on the KDF.
In [PBKDF2-HMAC](https://www.rfc-editor.org/rfc/rfc8018.html#section-5.2), the password serves as the HMAC key, while the salt followed by a block counter forms the first HMAC message.
The `"mnemonic" + BIP39_PASSPHRASE` expression builds the salt, which is passed separately to `hashlib.pbkdf2_hmac`.
For further reading, see [8.6 Key derivation with HKDF](https://livebook.manning.com/book/real-world-cryptography/chapter-8#102); HKDF is a separate KDF with its own construction.

The passphrase is a derivation input. Every different normalized passphrase gives another seed, without a wrong-password error.
Keep it available with the recovery phrase if you need to reconstruct this seed.
The 512-bit output does not increase the amount of randomness in the original input.

In [8]:
import unicodedata

password = unicodedata.normalize("NFKD", mnemonic).encode("utf-8")
salt = unicodedata.normalize(
    "NFKD", "mnemonic" + BIP39_PASSPHRASE
).encode("utf-8")

print(f"password = {password!r}")
print(f"salt = {salt!r}")

seed = hashlib.pbkdf2_hmac("sha512", password, salt, 2048, dklen=64)

print("Seed:", seed.hex())
print("Seed length (bytes):", len(seed))
print("Seed length (bits):", len(seed) * 8)

password = b'abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon abandon about'
salt = b'mnemonic'
Seed: 5eb00bbddcf069084889a8ab9155568165f5c453ccb85e70811aaed6f6da5fc19a5ac40b389cd370d086206dec8aa6c43daea6690f20ad3d8d48b2d2ce9e38e4
Seed length (bytes): 64
Seed length (bits): 512


## Compare with Published Test Vectors

The [reference vectors](https://github.com/trezor/python-mnemonic/blob/f5697ffc75547e0f2982b4f43dbd7f8810d09abd/vectors.json) use the optional passphrase `TREZOR`.
Use that same passphrase when comparing the two public mnemonics with the published seeds.
These checks are independent of the passphrase you selected for the main example and require no network access.

In [9]:
reference_vectors = [
    (
        MNEMONIC_12,
        "c55257c360c07c72029aebc1b53c05ed0362ada38ead3e3e9efa3708e5349553"
        "1f09a6987599d18264c1e1c92f2cf141630c7a3c4ab7c81b2f001698e7463b04",
    ),
    (
        MNEMONIC_24,
        "bda85446c68413707090a52022edd26a1c9462295029f2e60cd7c4f2bbd30971"
        "70af7a4d73245cafa9c3cca8d561a7c3de6f5d4a10be8ed2a5e608d68f92fcc8",
    ),
]

for reference_mnemonic, expected_seed in reference_vectors:
    reference_password = unicodedata.normalize("NFKD", reference_mnemonic).encode("utf-8")
    reference_salt = unicodedata.normalize("NFKD", "mnemonicTREZOR").encode("utf-8")
    reference_seed = hashlib.pbkdf2_hmac(
        "sha512", reference_password, reference_salt, 2048, dklen=64
    )
    assert reference_seed.hex() == expected_seed
    print(f"{len(reference_mnemonic.split())}-word published seed matches")

12-word published seed matches
24-word published seed matches


We now have the 64-byte BIP-39 `seed` and have verified the published examples.
The following cells use that seed as the input to BIP-32 key derivation.

## BIP-32: From Seed to Bitcoin and Ethereum Keys

[BIP-32](https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki) derives a tree of keys from one seed.
We derive Bitcoin native SegWit and Taproot keys, construct their receiving addresses, and follow an Ethereum key path with the same BIP-32 implementation.
Bitcoin's legacy, nested SegWit, native SegWit, and Taproot account conventions can coexist in one wallet:

| Bitcoin account type | Purpose | First receiving key path | Mainnet address prefix |
| --- | --- | --- | --- |
| Legacy P2PKH ([BIP-44](https://github.com/bitcoin/bips/blob/master/bip-0044.mediawiki)) | `44'` | `m/44'/0'/0'/0/0` | `1...` |
| Nested SegWit P2SH-P2WPKH ([BIP-49](https://github.com/bitcoin/bips/blob/master/bip-0049.mediawiki)) | `49'` | `m/49'/0'/0'/0/0` | `3...` |
| Native SegWit P2WPKH ([BIP-84](https://github.com/bitcoin/bips/blob/master/bip-0084.mediawiki)) | `84'` | `m/84'/0'/0'/0/0` | `bc1q...` |
| Taproot P2TR ([BIP-86](https://github.com/bitcoin/bips/blob/master/bip-0086.mediawiki)) | `86'` | `m/86'/0'/0'/0/0` | `bc1p...` |

### Reading the Path

The conventional structure is `m / purpose' / coin_type' / account' / change / address_index`.
Each slash represents one child derivation, starting from the master node.
For `m/84'/0'/0'/0/0`:

| Element | Role | Meaning in our Bitcoin example |
| --- | --- | --- |
| `m` | Master node | Starting point of the private key tree |
| `84'` | Purpose | BIP-84 native SegWit account convention |
| First `0'` | Coin type | Bitcoin mainnet |
| Second `0'` | Account | First account, numbered zero |
| First plain `0` | Change | External chain for receiving; `1` selects the internal change chain |
| Last `0` | Address index | First key on that chain; subsequent keys use `1`, `2`, and so on |

### Hardened Indices

An apostrophe marks a hardened index: add `2**31` to the displayed number.
Thus `84'` is `2147483732`, encoded as `80000054` in four-byte big-endian form.
`0'` becomes `80000000`, while normal `0` becomes `00000000`.

Hardened derivation uses parent private material.
Normal derivation supports deriving child public keys from a parent extended public key, enabling watch-only address generation.
Hardened account boundaries limit that public-only derivation and the parent-key exposure possible when a parent extended public key and a non-hardened child private key are both known.

In [10]:
BTC_PATH = "m/84'/0'/0'/0/0"
ETH_PATH = "m/44'/60'/0'/0/0"
HARDENED = 2 ** 31


def parse_path(path):
    parts = path.split("/")
    if parts[0] != "m":
        raise ValueError("Use an absolute private-key path starting with m")

    indices = []
    for part in parts[1:]:
        hardened = part.endswith("'")
        number = part[:-1] if hardened else part
        if not number.isascii() or not number.isdigit():
            raise ValueError(f"Invalid path element: {part!r}")
        index = int(number)
        if not 0 <= index < HARDENED:
            raise ValueError("Each path number must be between 0 and 2**31 - 1")
        indices.append(index + HARDENED if hardened else index)
    return indices


for label, index in zip(BTC_PATH.split("/")[1:], parse_path(BTC_PATH)):
    print(f"{label:>3}  integer={index:10d}  bytes={index.to_bytes(4, 'big').hex()}")

84'  integer=2147483732  bytes=80000054
 0'  integer=2147483648  bytes=80000000
 0'  integer=2147483648  bytes=80000000
  0  integer=         0  bytes=00000000
  0  integer=         0  bytes=00000000


### Master Private Key and Chain Code

Apply HMAC-SHA512 with the literal key `b"Bitcoin seed"` and our BIP-39 `seed` as the message.
Split its 64-byte result into a 32-byte private key and a 32-byte chain code.
The private scalar must lie between `1` and `n - 1`, where `n` is the secp256k1 generator's order.

The pair is an extended private key. Both halves are derived deterministically; no extra randomness is added.
The literal HMAC key is a public constant, while the seed carries the secret material.

In [11]:
import hmac

# Order of the secp256k1 generator point.
N = 0xFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFFEBAAEDCE6AF48A03BBFD25E8CD0364141


def master_from_seed(seed_bytes):
    if not 16 <= len(seed_bytes) <= 64:
        raise ValueError("BIP-32 seeds must contain 16 to 64 bytes")
    digest = hmac.new(b"Bitcoin seed", seed_bytes, hashlib.sha512).digest()
    private_key, chain_code = digest[:32], digest[32:]
    if not 1 <= int.from_bytes(private_key, "big") < N:
        raise ValueError("Invalid BIP-32 master key")
    return private_key, chain_code


master_private_key, master_chain_code = master_from_seed(seed)
print("HMAC key:", b"Bitcoin seed")
print("HMAC message (seed):", seed.hex())
print("Master private key (I_L):", master_private_key.hex())
print("Master chain code (I_R):", master_chain_code.hex())

HMAC key: b'Bitcoin seed'
HMAC message (seed): 5eb00bbddcf069084889a8ab9155568165f5c453ccb85e70811aaed6f6da5fc19a5ac40b389cd370d086206dec8aa6c43daea6690f20ad3d8d48b2d2ce9e38e4
Master private key (I_L): 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
Master chain code (I_R): 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e


### The Public-Key Primitive

Normal child derivation needs the parent's compressed public key: a 33-byte encoding of the secp256k1 point obtained by multiplying the generator by the private scalar.
The first byte is `02` or `03`, identifying the parity of the y coordinate; the remaining 32 bytes encode x.

We use the low-level [`coincurve`](https://github.com/ofek/coincurve) package, pinned to version `21.0.0`, for that operation.
The BIP-32 HMAC and scalar arithmetic below are implemented explicitly.

In [12]:
from coincurve import PublicKey


def compressed_public_key(private_key):
    return PublicKey.from_secret(private_key).format(compressed=True)


print("Master public key:", compressed_public_key(master_private_key).hex())

Master public key: 03d902f35f560e0470c63313c7369168d9d7df2d49bf295fd9fb7cb109ccee0494


### Deriving a Child Private Key

Use the parent chain code as the HMAC-SHA512 key.
Its 37-byte message has the following layout:

| Derivation | First 33 message bytes | Last four message bytes |
| --- | --- | --- |
| Hardened | `0x00` followed by the 32-byte parent private key | Encoded child index, including its hardened bit |
| Normal | The 33-byte compressed parent public key | Encoded child index |

Split the result into `I_L` and `I_R`.
The child private scalar is `(parent_scalar + integer(I_L)) % n`; `I_R` is the new chain code.
Encode the scalar as exactly 32 big-endian bytes, retaining leading zeroes.

Reject `integer(I_L) >= n` and a zero child scalar.
BIP-32 specifies trying the next index for an invalid result; this helper raises an error so that an updated path can be chosen and recorded explicitly.
`trace=True` prints the inputs and intermediate values.

In [13]:
def derive_child(parent_private_key, parent_chain_code, index, trace=False):
    if len(parent_private_key) != 32 or len(parent_chain_code) != 32:
        raise ValueError("Private keys and chain codes must each contain 32 bytes")
    parent_scalar = int.from_bytes(parent_private_key, "big")
    if not 1 <= parent_scalar < N:
        raise ValueError("Invalid parent private key")
    if not 0 <= index < 2 ** 32:
        raise ValueError("Child index must fit into four unsigned bytes")

    hardened = index >= HARDENED
    index_bytes = index.to_bytes(4, "big")
    if hardened:
        parent_data = b"\x00" + parent_private_key
    else:
        parent_data = compressed_public_key(parent_private_key)
    data = parent_data + index_bytes

    digest = hmac.new(parent_chain_code, data, hashlib.sha512).digest()
    left, child_chain_code = digest[:32], digest[32:]
    tweak = int.from_bytes(left, "big")
    if tweak >= N:
        raise ValueError("Invalid BIP-32 child; try the next index")
    child_scalar = (parent_scalar + tweak) % N
    if child_scalar == 0:
        raise ValueError("Invalid BIP-32 child; try the next index")
    child_private_key = child_scalar.to_bytes(32, "big")

    if trace:
        print("Hardened:", hardened)
        print("Child index:", index)
        print("Index bytes:", index_bytes.hex())
        print("HMAC key (parent chain code):", parent_chain_code.hex())
        print("HMAC message:", data.hex())
        print("I_L:", left.hex())
        print("I_R (child chain code):", child_chain_code.hex())
        print("Parent private key:", parent_private_key.hex())
        print("Child private key:", child_private_key.hex())

    return child_private_key, child_chain_code

### First Hardened Step: m to m/84'

The encoded index is `80000054`.
The message begins with a zero byte and the parent private key, followed by that four-byte index.

In [14]:
purpose_private_key, purpose_chain_code = derive_child(
    master_private_key, master_chain_code, HARDENED + 84, trace=True
)

Hardened: True
Child index: 2147483732
Index bytes: 80000054
HMAC key (parent chain code): 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e
HMAC message: 001837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf6780000054
I_L: db3254aa9427b8422ab16730d39b931c33e8342267beb5b952ea74cc3f856e35
I_R (child chain code): ca6a9276f135e836a2c3222f4901cc0f323adcbb39f746ef3c1ab2b7460209e0
Parent private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
Child private key: f36a166922514e2e3c7f09e139b0aeff039cbf020606670b27554f7fe1a24d9c


### First Normal Step: the Receiving Chain

Two further hardened steps select Bitcoin and account zero.
Then index `0` selects the external chain at `m/84'/0'/0'/0`.
This message begins with the account's compressed public key.

In [15]:
coin_private_key, coin_chain_code = derive_child(
    purpose_private_key, purpose_chain_code, HARDENED + 0
)
account_private_key, account_chain_code = derive_child(
    coin_private_key, coin_chain_code, HARDENED + 0
)
external_private_key, external_chain_code = derive_child(
    account_private_key, account_chain_code, 0, trace=True
)

Hardened: False
Child index: 0
Index bytes: 00000000
HMAC key (parent chain code): 4a53a0ab21b9dc95869c4e92a161194e03c0ef3ff5014ac692f433c4765490fc
HMAC message: 02707a62fdacc26ea9b63b1c197906f56ee0180d0bcf1966e1a2da34f5f3a09a9b00000000
I_L: 9b04ef55756992a33846cef1e05aa1d2a72b8dae62aea873666893a103fc51fd
I_R (child chain code): 5fb17f32471143583da296b3af79da1a98f63db12389a0150a67610620925a38
Parent private key: e14f274d16ca0d91031b98b162618061d03930fa381af6d4caf44b01819ab6d4
Child private key: 7c5416a28c33a0343b6267a342bc2235bcb5e1c1eb80ff0c718a8015b560c790


### Walk the Complete Bitcoin Path

Repeat the same child operation at every level, retaining the resulting key and chain code for the next step.
Print each node and finish with the first receiving private key and its compressed public key.

In [16]:
def derive_path(root_private_key, root_chain_code, path, trace=False):
    private_key, chain_code = root_private_key, root_chain_code
    current_path = "m"
    nodes = [(current_path, private_key, chain_code)]

    for index in parse_path(path):
        private_key, chain_code = derive_child(private_key, chain_code, index)
        label = f"{index - HARDENED}'" if index >= HARDENED else str(index)
        current_path += "/" + label
        nodes.append((current_path, private_key, chain_code))

    if trace:
        for node_path, node_private_key, node_chain_code in nodes:
            print(node_path)
            print("  private key:", node_private_key.hex())
            print("  chain code:", node_chain_code.hex())
    return private_key, chain_code

In [17]:
btc_private_key, btc_chain_code = derive_path(
    master_private_key, master_chain_code, BTC_PATH, trace=True
)
btc_public_key = compressed_public_key(btc_private_key)
print("Bitcoin private key:", btc_private_key.hex())
print("Bitcoin public key:", btc_public_key.hex())

m
  private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
  chain code: 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e
m/84'
  private key: f36a166922514e2e3c7f09e139b0aeff039cbf020606670b27554f7fe1a24d9c
  chain code: ca6a9276f135e836a2c3222f4901cc0f323adcbb39f746ef3c1ab2b7460209e0
m/84'/0'
  private key: fe87dbc6d34a7e2aca1d727d3922eaf7c21a7f7c95bb8858dd8429a8a69c14f4
  chain code: 6c2d070f9d27f0f5c7900f164b4037e09301826bb874202f2f1c8095413d36bb
m/84'/0'/0'
  private key: e14f274d16ca0d91031b98b162618061d03930fa381af6d4caf44b01819ab6d4
  chain code: 4a53a0ab21b9dc95869c4e92a161194e03c0ef3ff5014ac692f433c4765490fc
m/84'/0'/0'/0
  private key: 7c5416a28c33a0343b6267a342bc2235bcb5e1c1eb80ff0c718a8015b560c790
  chain code: 5fb17f32471143583da296b3af79da1a98f63db12389a0150a67610620925a38
m/84'/0'/0'/0/0
  private key: 4604b4b710fe91f584fff084e1a9159fe4f8408fff380596a604948474ce4fa3
  chain code: ab52cce8c3dc905c885ff8aa6d3374df3c27a38363acd3d62379

### Encoding Bech32 and Bech32m Addresses

We now have a derived public key, but the receiving address still needs to be constructed.
For both native SegWit and Taproot, the final text-encoding step uses the network prefix, witness version, destination bytes, and a checksum.
We will implement that encoding once and use it for both address types.

First, regroup bytes into five-bit values.
This resembles our earlier conversion to 11-bit word-list indices, except that each group now selects one of 32 characters.
Zero bits pad the final group when necessary:

In [18]:
def five_bit_values(raw_bytes):
    bits = "".join(f"{byte:08b}" for byte in raw_bytes)
    bits += "0" * (-len(bits) % 5)
    return [int(bits[offset:offset + 5], 2) for offset in range(0, len(bits), 5)]

The checksum uses the polynomial recurrence specified by [BIP-173](https://github.com/bitcoin/bips/blob/master/bip-0173.mediawiki).
The five generator constants below define that calculation; they are public format parameters.
The network prefix is split into high and low character bits, with a zero between them, so it is also covered by the checksum.

We append six zero values while calculating the checksum, then apply the format's final constant.
That constant is `1` for Bech32 and `0x2BC830A3` for Bech32m, following [BIP-350](https://github.com/bitcoin/bips/blob/master/bip-0350.mediawiki).
The resulting 30 bits become six five-bit checksum characters:

In [19]:
BECH32_ALPHABET = "qpzry9x8gf2tvdw0s3jn54khce6mua7l"


def bech32_polymod(values):
    generators = (
        0x3B6A57B2, 0x26508E6D, 0x1EA119FA, 0x3D4233DD, 0x2A1462B3,
    )
    remainder = 1
    for value in values:
        high = remainder >> 25
        remainder = ((remainder & 0x1FFFFFF) << 5) ^ value
        for bit, generator in enumerate(generators):
            if (high >> bit) & 1:
                remainder ^= generator
    return remainder


def segwit_address(hrp, witness_version, witness_program):
    if not hrp or hrp != hrp.lower() or any(not 33 <= ord(c) <= 126 for c in hrp):
        raise ValueError("Use a nonempty lowercase printable network prefix")
    if not 0 <= witness_version <= 16:
        raise ValueError("Witness version must be between 0 and 16")
    if not 2 <= len(witness_program) <= 40:
        raise ValueError("Witness program must contain 2 to 40 bytes")
    if witness_version == 0 and len(witness_program) not in (20, 32):
        raise ValueError("A version-0 witness program must contain 20 or 32 bytes")

    data = [witness_version] + five_bit_values(witness_program)
    expanded_hrp = [ord(c) >> 5 for c in hrp] + [0] + [ord(c) & 31 for c in hrp]
    checksum_constant = 1 if witness_version == 0 else 0x2BC830A3
    remainder = bech32_polymod(expanded_hrp + data + [0] * 6) ^ checksum_constant
    checksum = [(remainder >> (5 * position)) & 31 for position in range(5, -1, -1)]
    address = hrp + "1" + "".join(BECH32_ALPHABET[value] for value in data + checksum)
    if len(address) > 90:
        raise ValueError("A Bech32 or Bech32m address cannot exceed 90 characters")
    return address

The first data value is the witness version, which explains why the alphabet produces `q` for version `0` and `p` for version `1`.
The destination bytes and checksum follow it.
For our Bitcoin mainnet examples, the readable network prefix is `bc`.

### Constructing a BIP-84 Address

A P2WPKH address uses the **20-byte public-key hash** as its witness program.
This hash is called **HASH160**: apply SHA-256 to the compressed public key, then RIPEMD-160 to that 32-byte digest.
The result is public destination data.
Encoding it with witness version `0` selects Bech32:

In [20]:
btc_key_hash = hashlib.new("ripemd160", hashlib.sha256(btc_public_key).digest()).digest()
btc_address = segwit_address("bc", 0, btc_key_hash)

print("Compressed public key:", btc_public_key.hex())
print("HASH160 witness program:", btc_key_hash.hex())
print("BIP-84 address:", btc_address)

Compressed public key: 0330d54fd0dd420a6e5f8d3624f5f3482cae350f79d5f0753bf5beef9c2d91af3c
HASH160 witness program: c0cebcd6c3d3ca8c75dc5ec62ebe55330ef910e2
BIP-84 address: bc1qcr8te4kr609gcawutmrza0j4xv80jy8z306fyu


For the public 12-word example with an empty passphrase, the address is `bc1qcr8te4kr609gcawutmrza0j4xv80jy8z306fyu`, matching BIP-84's first receiving address.

### Constructing a Taproot Address with BIP-86

Now we can return to the extra step mentioned in the account-type discussion.
[BIP-86](https://github.com/bitcoin/bips/blob/master/bip-0086.mediawiki) starts with a different BIP-32 path, `m/86'/0'/0'/0/0`.
The derived key supplies the Taproot **internal key**.
We print its private key and compressed public key so that we can see exactly where ordinary BIP-32 derivation ends:

In [21]:
TAPROOT_PATH = "m/86'/0'/0'/0/0"
taproot_private_key, taproot_chain_code = derive_path(
    master_private_key, master_chain_code, TAPROOT_PATH
)
taproot_public_key = compressed_public_key(taproot_private_key)
internal_key = taproot_public_key[1:]

print("Taproot path:", TAPROOT_PATH)
print("Derived internal private key:", taproot_private_key.hex())
print("Derived compressed public key:", taproot_public_key.hex())
print("Internal key, x coordinate only:", internal_key.hex())

Taproot path: m/86'/0'/0'/0/0
Derived internal private key: 41f41d69260df4cf277826a9b65a3717e4eeddbeedf637f212ca096576479361
Derived compressed public key: 03cc8a4bc64d897bddc5fbc2f670f7a8ba0b386779106cf1223c6fc5d7cd6fc115
Internal key, x coordinate only: cc8a4bc64d897bddc5fbc2f670f7a8ba0b386779106cf1223c6fc5d7cd6fc115


Taproot uses **x-only public keys**, encoded as 32-byte x coordinates.
An x coordinate corresponds to two curve points with opposite y coordinates.
The convention selects the point with **even y**, called `P` below.
We obtain that point by parsing `0x02` followed by the x coordinate; the `02` compressed-key prefix specifies even y.
If the original compressed key began with `03`, this selects the opposite point with the same x coordinate.

Next comes the **TapTweak**, defined by [BIP-341](https://github.com/bitcoin/bips/blob/master/bip-0341.mediawiki).
A tagged SHA-256 hash prefixes its message with two copies of the tag's SHA-256 digest:

```text
tag_hash = SHA256(b"TapTweak")
tweak_bytes = SHA256(tag_hash || tag_hash || internal_key)
t = integer(tweak_bytes)
Q = P + t * G
```

Here, `||` means byte concatenation, `G` is the secp256k1 generator, and `t` is interpreted as a big-endian integer.
The tag separates this hash's purpose from other uses of SHA-256.
BIP-86 supplies no script-tree Merkle root, so the message after the two tag hashes contains only the internal key's 32 bytes.
There is no extra all-zero root appended.

The scalar must be smaller than `n`; an out-of-range digest is rejected, rather than reduced modulo `n`.
The result `Q` must also be a valid point.
We use `coincurve` only for lifting the x coordinate and calculating `P + t * G`; the hash inputs and address encoding remain visible in our code:

In [22]:
def tagged_hash(tag, message):
    tag_digest = hashlib.sha256(tag).digest()
    return hashlib.sha256(tag_digest + tag_digest + message).digest()


def taproot_output_key(internal_key, trace=False):
    if len(internal_key) != 32:
        raise ValueError("The internal key must be a 32-byte x coordinate")
    internal_point = PublicKey(b"\x02" + internal_key)
    tweak_bytes = tagged_hash(b"TapTweak", internal_key)
    if int.from_bytes(tweak_bytes, "big") >= N:
        raise ValueError("TapTweak is outside the valid scalar range")

    # PublicKey.add(t) computes P + t * G and rejects a point at infinity.
    output_point = internal_point.add(tweak_bytes)
    compressed_output = output_point.format(compressed=True)
    output_key = compressed_output[1:]

    if trace:
        tag_digest = hashlib.sha256(b"TapTweak").digest()
        print("Even-y internal point P:", internal_point.format(compressed=True).hex())
        print("SHA256(TapTweak):", tag_digest.hex())
        print("Complete hash input:", (tag_digest + tag_digest + internal_key).hex())
        print("TapTweak bytes:", tweak_bytes.hex())
        print("Output point Q:", compressed_output.hex())
        print("Output key, x coordinate only:", output_key.hex())
    return tweak_bytes, output_key

The **32-byte x coordinate of `Q`**, rather than the internal key, is the Taproot witness program.
Encoding that program with witness version `1` selects Bech32m.
For comparison, the on-chain output script, or `scriptPubKey`, starts with `0x51` for `OP_1`, then `0x20` to push 32 bytes:

In [23]:
taproot_tweak, taproot_program = taproot_output_key(internal_key, trace=True)
taproot_script = b"\x51\x20" + taproot_program
taproot_address = segwit_address("bc", 1, taproot_program)

print("Taproot scriptPubKey:", taproot_script.hex())
print("BIP-86 address:", taproot_address)

Even-y internal point P: 02cc8a4bc64d897bddc5fbc2f670f7a8ba0b386779106cf1223c6fc5d7cd6fc115
SHA256(TapTweak): e80fe1639c9ca050e3af1b39c143c63e429cbceb15d940fbb5c5a1f4af57c5e9
Complete hash input: e80fe1639c9ca050e3af1b39c143c63e429cbceb15d940fbb5c5a1f4af57c5e9e80fe1639c9ca050e3af1b39c143c63e429cbceb15d940fbb5c5a1f4af57c5e9cc8a4bc64d897bddc5fbc2f670f7a8ba0b386779106cf1223c6fc5d7cd6fc115
TapTweak bytes: 2ca01ed85cf6b6526f73d39a1111cd80333bfdc00ce98992859848a90a6f0258
Output point Q: 03a60869f0dbcf1dc659c9cecbaf8050135ea9e8cdc487053f1dc6880949dc684c
Output key, x coordinate only: a60869f0dbcf1dc659c9cecbaf8050135ea9e8cdc487053f1dc6880949dc684c
Taproot scriptPubKey: 5120a60869f0dbcf1dc659c9cecbaf8050135ea9e8cdc487053f1dc6880949dc684c
BIP-86 address: bc1p5cyxnuxmeuwuvkwfem96lqzszd02n6xdcjrs20cac6yqjjwudpxqkedrcr


For the public 12-word example with an empty passphrase, the intermediate and final values are:

```text
Internal key: cc8a4bc64d897bddc5fbc2f670f7a8ba0b386779106cf1223c6fc5d7cd6fc115
Output key:   a60869f0dbcf1dc659c9cecbaf8050135ea9e8cdc487053f1dc6880949dc684c
Address:      bc1p5cyxnuxmeuwuvkwfem96lqzszd02n6xdcjrs20cac6yqjjwudpxqkedrcr
```

The internal key and output key differ even though both come from the same recovery inputs.
The BIP-32 path selects the internal key; the TapTweak produces the output key; Bech32m writes the resulting public destination as address text.
The notebook checks this complete sequence against the published BIP-86 examples and a BIP-341 wallet test vector.

#### Deriving the Taproot Signing Key

The address calculation gives us the public output key `Q`.
For a key-path spend, we also need its **tweaked private key**, as specified by [BIP-341](https://github.com/bitcoin/bips/blob/master/bip-0341.mediawiki#constructing-and-spending-taproot-outputs).

Let `d0` be the private scalar derived by BIP-32.
Our internal point `P` was selected to have even y.
If `d0 * G` has odd y, the scalar for `P` is therefore `N - d0`; otherwise it remains `d0`.
Negating the scalar negates the point, preserving x and reversing y.
We then add the same TapTweak scalar `t` that we used on the public side:

```text
d = d0                  if d0 * G has even y
d = N - d0              if d0 * G has odd y
q = (d + t) mod N

q * G = P + t * G = Q
```

The 32-byte encoding of `q` is the tweaked private key.
We check the same tweak bound as before and reject zero, which would produce the point at infinity:


In [24]:
def taproot_output_private_key(internal_private_key, trace=False):
    public_key = compressed_public_key(internal_private_key)
    d0 = int.from_bytes(internal_private_key, "big")
    d = d0 if public_key[0] == 2 else N - d0
    tweak_bytes = tagged_hash(b"TapTweak", public_key[1:])
    t = int.from_bytes(tweak_bytes, "big")
    if t >= N:
        raise ValueError("TapTweak is outside the valid scalar range")
    q = (d + t) % N
    if q == 0:
        raise ValueError("TapTweak produced an invalid private key")
    output_private_key = q.to_bytes(32, "big")

    if trace:
        print("Derived BIP-32 point has odd y:", public_key[0] == 3)
        print("Even-y internal scalar d:", d.to_bytes(32, "big").hex())
        print("TapTweak scalar t:", tweak_bytes.hex())
        print("Tweaked private scalar q:", output_private_key.hex())
    return output_private_key


In [25]:
taproot_signing_private_key = taproot_output_private_key(
    taproot_private_key, trace=True
)
signing_public_key = compressed_public_key(taproot_signing_private_key)
assert signing_public_key[1:] == taproot_program
print("Signing public key matches the output key:", signing_public_key[1:].hex())


Derived BIP-32 point has odd y: True
Even-y internal scalar d: be0be296d9f20b30d887d95649a5c8e6d5bfff27c1526849ad08552759eeade0
TapTweak scalar t: 2ca01ed85cf6b6526f73d39a1111cd80333bfdc00ce98992859848a90a6f0258
Tweaked private scalar q: eaac016f36e8c18347fbacf05ab7966708fbfce7ce3bf1dc32a09dd0645db038
Signing public key matches the output key: a60869f0dbcf1dc659c9cecbaf8050135ea9e8cdc487053f1dc6880949dc684c


This verifies that the private calculation reaches the same output key as the earlier public calculation.
For a Bitcoin key-path spend, the wallet signs the transaction's Taproot signature hash with this tweaked private key.
`Q` itself can have odd y; BIP-340 Schnorr signing applies its own even-y normalization to the tweaked key, which `coincurve.PrivateKey.sign_schnorr()` handles for us.
The notebook checks the tweaked private key and resulting Schnorr signature against BIP-341's published spending vector.


### Follow the Ethereum Path

For our Ethereum example, use `m/44'/60'/0'/0/0`.
BIP-44 supplies the hierarchy; [SLIP-44](https://github.com/satoshilabs/slips/blob/master/slip-0044.md) assigns Ethereum coin type `60`.
The underlying BIP-32 algorithm and the master node, including the literal `b"Bitcoin seed"` input, remain the same.
The different purpose and coin-type indices lead to another branch.

The compressed public key printed here is used by BIP-32; Ethereum address construction is a further operation.

In [26]:
eth_private_key, eth_chain_code = derive_path(
    master_private_key, master_chain_code, ETH_PATH, trace=True
)
eth_public_key = compressed_public_key(eth_private_key)
print("Ethereum private key:", eth_private_key.hex())
print("Ethereum compressed public key:", eth_public_key.hex())

m
  private key: 1837c1be8e2995ec11cda2b066151be2cfb48adf9e47b151d46adab3a21cdf67
  chain code: 7923408dadd3c7b56eed15567707ae5e5dca089de972e07f3b860450e2a3b70e
m/44'
  private key: cd46049fb82a4fccbd967c5234e61152f4df87d01d1ef833420814c67ee4b7ea
  chain code: 45d3b0e8206db10a08d555317c7e245c5bbd12254ce968f3c79a959d4e6af98a
m/44'/60'
  private key: eea62fd3712ad7ff27f7145280d07f0041a51ef2f35432d6b488cb294cf79bef
  chain code: ff0603122d4dc5bf9b36606ef1b1b6e5a6c9e79fc6a4b08f7e871eee10457a71
m/44'/60'/0'
  private key: e9b91c428e679f857fbfa0203440eaeaabb0ca46d80e70b6aa6250696ac0dfc7
  chain code: d882718b7a42806803eeb17f7483f20620611adb88fc943c898dc5aba94c2819
m/44'/60'/0'/0
  private key: a29ac2cb17e31cdab42a8fe2d83f04f4b69c5e73bc8d3bf6b5dc96ac239b145a
  chain code: c879d136f02003dc804811e42390714ea06448aeaee158c62d65f12a421d988f
m/44'/60'/0'/0/0
  private key: 1ab42cc412b618bdea3a599e3c9bae199ebf030895b039e9db1e30dafb12b727
  chain code: 736094f4f24b67e838a4b3d23d31d229ca03e00c9bb99ce9

### Check Published BIP-32, BIP-84, and Taproot Vectors

The [BIP-32 vectors](https://github.com/bitcoin/bips/blob/master/bip-0032.mediawiki#test-vectors) below check a master node, a mixed hardened/normal path, and a leading-zero case.
The expected private keys and chain codes are the raw fields from the published extended private keys.

The [BIP-84 examples](https://github.com/bitcoin/bips/blob/master/bip-0084.mediawiki#test-vectors) check two receiving keys and one change key.
Their expected raw private bytes correspond to the published WIF values.
The reference mnemonic is always the public 12-word phrase with an empty passphrase, independent of the main example's selected inputs.

We also verify all three [BIP-86 examples](https://github.com/bitcoin/bips/blob/master/bip-0086.mediawiki#test-vectors): the x-only internal key, tweaked output key, and final address.
A separate [BIP-341 wallet vector](https://github.com/bitcoin/bips/blob/master/bip-0341/wallet-test-vectors.json) checks the exact TapTweak digest, output key, output script, and Bech32m address for a supplied internal key without a script tree.
The matching key-path spending vector also supplies the internal private key, expected tweaked private key, precomputed transaction signature hash, and expected Schnorr signature.
We reproduce that signature with the vector's all-zero auxiliary randomness and verify it against the output key.
These fixed examples are independent of the main example's entropy length and passphrase.

In [27]:
bip32_reference_nodes = [
    (
        "000102030405060708090a0b0c0d0e0f",
        "m",
        "e8f32e723decf4051aefac8e2c93c9c5b214313817cdb01a1494b917c8436b35",
        "873dff81c02f525623fd1fe5167eac3a55a049de3d314bb42ee227ffed37d508",
    ),
    (
        "000102030405060708090a0b0c0d0e0f",
        "m/0'/1/2'/2/1000000000",
        "471b76e389e528d6de6d816857e012c5455051cad6660850e58372a6c3e6e7c8",
        "c783e67b921d2beb8f6b389cc646d7263b4145701dadd2161548a8b078e65e9e",
    ),
    (
        "3ddd5602285899a946114506157c7997e5444528f3003f6134712147db19b678",
        "m/0'/1'",
        "3a2086edd7d9df86c3487a5905a1712a9aa664bce8cc268141e07549eaa8661d",
        "a48ee6674c5264a237703fd383bccd9fad4d9378ac98ab05e6e7029b06360c0d",
    ),
]

for seed_hex, path, expected_private, expected_chain in bip32_reference_nodes:
    reference_root = master_from_seed(bytes.fromhex(seed_hex))
    actual_private, actual_chain = derive_path(*reference_root, path)
    assert actual_private.hex() == expected_private
    assert actual_chain.hex() == expected_chain
    print("Published BIP-32 node matches:", path)

Published BIP-32 node matches: m
Published BIP-32 node matches: m/0'/1/2'/2/1000000000
Published BIP-32 node matches: m/0'/1'


In [28]:
# The published BIP-84 examples use the 12-word phrase and an empty passphrase.
reference_seed = hashlib.pbkdf2_hmac(
    "sha512",
    unicodedata.normalize("NFKD", MNEMONIC_12).encode("utf-8"),
    b"mnemonic",
    2048,
    dklen=64,
)
reference_root = master_from_seed(reference_seed)
bip84_reference_nodes = [
    (
        "m/84'/0'/0'/0/0",
        "4604b4b710fe91f584fff084e1a9159fe4f8408fff380596a604948474ce4fa3",
        "0330d54fd0dd420a6e5f8d3624f5f3482cae350f79d5f0753bf5beef9c2d91af3c",
    ),
    (
        "m/84'/0'/0'/0/1",
        "2fd0affa51529f940a358ec0c50de81267d0bf5158ca61887347676946362c5b",
        "03e775fd51f0dfb8cd865d9ff1cca2a158cf651fe997fdc9fee9c1d3b5e995ea77",
    ),
    (
        "m/84'/0'/0'/1/0",
        "3277578a56b721e4c9f071f1e24aa0f94c4ff72e7967fea03b134f605f07c8fd",
        "03025324888e429ab8e3dbaf1f7802648b9cd01e9b418485c5fa4c1b9b5700e1a6",
    ),
]

bip84_reference_addresses = {"m/84'/0'/0'/0/0": 'bc1qcr8te4kr609gcawutmrza0j4xv80jy8z306fyu',
 "m/84'/0'/0'/0/1": 'bc1qnjg0jd8228aq7egyzacy8cys3knf9xvrerkf9g',
 "m/84'/0'/0'/1/0": 'bc1q8c6fshw2dlwun7ekn9qwf37cu2rn755upcp6el'}

for path, expected_private, expected_public in bip84_reference_nodes:
    actual_private, _ = derive_path(*reference_root, path)
    assert actual_private.hex() == expected_private
    assert compressed_public_key(actual_private).hex() == expected_public
    reference_hash = hashlib.new("ripemd160", hashlib.sha256(compressed_public_key(actual_private)).digest()).digest()
    actual_address = segwit_address("bc", 0, reference_hash)
    assert actual_address == bip84_reference_addresses[path]
    print("Published BIP-84 key and address match:", path, actual_address)

Published BIP-84 key and address match: m/84'/0'/0'/0/0 bc1qcr8te4kr609gcawutmrza0j4xv80jy8z306fyu
Published BIP-84 key and address match: m/84'/0'/0'/0/1 bc1qnjg0jd8228aq7egyzacy8cys3knf9xvrerkf9g
Published BIP-84 key and address match: m/84'/0'/0'/1/0 bc1q8c6fshw2dlwun7ekn9qwf37cu2rn755upcp6el


In [29]:
bip86_reference_nodes = [("m/86'/0'/0'/0/0",
  'cc8a4bc64d897bddc5fbc2f670f7a8ba0b386779106cf1223c6fc5d7cd6fc115',
  'a60869f0dbcf1dc659c9cecbaf8050135ea9e8cdc487053f1dc6880949dc684c',
  'bc1p5cyxnuxmeuwuvkwfem96lqzszd02n6xdcjrs20cac6yqjjwudpxqkedrcr'),
 ("m/86'/0'/0'/0/1",
  '83dfe85a3151d2517290da461fe2815591ef69f2b18a2ce63f01697a8b313145',
  'a82f29944d65b86ae6b5e5cc75e294ead6c59391a1edc5e016e3498c67fc7bbb',
  'bc1p4qhjn9zdvkux4e44uhx8tc55attvtyu358kutcqkudyccelu0was9fqzwh'),
 ("m/86'/0'/0'/1/0",
  '399f1b2f4393f29a18c937859c5dd8a77350103157eb880f02e8c08214277cef',
  '882d74e5d0572d5a816cef0041a96b6c1de832f6f9676d9605c44d5e9a97d3dc',
  'bc1p3qkhfews2uk44qtvauqyr2ttdsw7svhkl9nkm9s9c3x4ax5h60wqwruhk7')]

for path, expected_internal, expected_output, expected_address in bip86_reference_nodes:
    reference_private, _ = derive_path(*reference_root, path)
    actual_internal = compressed_public_key(reference_private)[1:]
    actual_tweak, actual_output = taproot_output_key(actual_internal)
    actual_address = segwit_address("bc", 1, actual_output)
    assert actual_internal.hex() == expected_internal
    assert actual_output.hex() == expected_output
    assert actual_address == expected_address
    actual_signing_private = taproot_output_private_key(reference_private)
    assert compressed_public_key(actual_signing_private)[1:] == actual_output
    # Opposite original point parity must select the same even-y internal point.
    negated_private = (N - int.from_bytes(reference_private, "big")).to_bytes(32, "big")
    assert taproot_output_private_key(negated_private) == actual_signing_private
    print("Published BIP-86 internal key, output key, signing key, and address match:", path)

Published BIP-86 internal key, output key, signing key, and address match: m/86'/0'/0'/0/0
Published BIP-86 internal key, output key, signing key, and address match: m/86'/0'/0'/0/1
Published BIP-86 internal key, output key, signing key, and address match: m/86'/0'/0'/1/0


In [30]:
# BIP-341's first scriptPubKey vector has no script tree.
reference_internal = bytes.fromhex('d6889cb081036e0faefa3a35157ad71086b123b2b144b649798b494c300a961d')
actual_tweak, actual_output = taproot_output_key(reference_internal)
assert actual_tweak.hex() == 'b86e7be8f39bab32a6f2c0443abbc210f0edac0e2c53d501b36b64437d9c6c70'
assert actual_output.hex() == '53a1f6e454df1aa2776a2814a721372d6258050de330b3c6d10ee8f4e0dda343'
assert (b"\x51\x20" + actual_output).hex() == '512053a1f6e454df1aa2776a2814a721372d6258050de330b3c6d10ee8f4e0dda343'
assert segwit_address("bc", 1, actual_output) == 'bc1p2wsldez5mud2yam29q22wgfh9439spgduvct83k3pm50fcxa5dps59h4z5'
print("Published BIP-341 tweak, output key, script, and address match.")

Published BIP-341 tweak, output key, script, and address match.


In [31]:
from coincurve import PrivateKey, PublicKeyXOnly

# BIP-341's first key-path spending vector uses the same script-free output.
reference_internal_private = bytes.fromhex("6b973d88838f27366ed61c9ad6367663045cb456e28335c109e30717ae0c6baa")
reference_signing_private = taproot_output_private_key(reference_internal_private)
reference_signing_key = PrivateKey(reference_signing_private)
reference_output = actual_output
assert reference_signing_private.hex() == "2405b971772ad26915c8dcdf10f238753a9b837e5f8e6a86fd7c0cce5b7296d9"
assert reference_signing_key.public_key_xonly.format() == reference_output

reference_sighash = bytes.fromhex("2514a6272f85cfa0f45eb907fcb0d121b808ed37c6ea160a5a9046ed5526d555")
reference_signature = reference_signing_key.sign_schnorr(
    reference_sighash, aux_randomness=bytes(32)
)
assert reference_signature.hex() == (
    "ed7c1647cb97379e76892be0cacff57ec4a7102aa24296ca39af7541246d8ff1"
    "4d38958d4cc1e2e478e4d4a764bbfd835b16d4e314b72937b29833060b87276c"
)
assert PublicKeyXOnly(reference_output).verify(reference_signature, reference_sighash)
print("Published BIP-341 tweaked private key and Schnorr signature match.")


Published BIP-341 tweaked private key and Schnorr signature match.


We have derived Bitcoin and Ethereum keys and constructed Bitcoin native SegWit and Taproot receiving addresses.
The Taproot example shows the x-only internal key, tagged-hash input, TapTweak, output point, witness program, final Bech32m address, and matching tweaked private key.
Its private-key calculation and Schnorr signing reproduce the published BIP-341 spending example.
To reconstruct these results, retain the recovery phrase, any BIP-39 passphrase, derivation path, and account convention.
Ethereum address encoding is a separate procedure beyond the key derivation shown here.